In [0]:
%sql
INSERT INTO pyspark_data.source.customers (id, email, city, country, modified_date) VALUES
(1, 'abc223@example.com', 'seattle', 'USA', current_timestamp()),
(6, 'qwejones@example.com', 'London', 'UK', current_timestamp())

In [0]:
# Create the customers table
spark.sql("""
CREATE TABLE pyspark_data.source.customers (
    id INT,
    email STRING,
    city STRING,
    country STRING,
    modified_date TIMESTAMP
)
""")

# Insert values into the customers table
spark.sql("""
INSERT INTO pyspark_data.source.customers (id, email, city, country, modified_date) VALUES
(1, 'john.doe@example.com', 'New York', 'USA', current_timestamp()),
(2, 'jane.smith@example.com', 'Los Angeles', 'USA', current_timestamp()),
(3, 'alice.jones@example.com', 'London', 'UK', current_timestamp())
""")

In [0]:
%sql
select * from pyspark_data.source.customers

### **_create table if not exists_**

In [0]:
# Create the DimCustomers table
spark.sql("""
CREATE TABLE pyspark_data.source.DimCustomers (
    id INT,
    email STRING,
    city STRING,
    country STRING,
    modified_date TIMESTAMP,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    is_active STRING
)
""")

# Insert values into the DimCustomers table
spark.sql("""
INSERT INTO pyspark_data.source.DimCustomers (id, email, city, country, modified_date, start_time, end_time, is_active)
SELECT 
    id, 
    email, 
    city, 
    country, 
    modified_date, 
    current_timestamp() AS start_time, 
    '9999-12-31 23:59:59' AS end_time, 
    'Y' AS is_active
FROM pyspark_data.source.customers
""")

In [0]:
%sql
select * from pyspark_data.source.dimcustomers

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

df = spark.sql(
    """
    SELECT * FROM pyspark_data.source.customers
    """
)

df = df.withColumn(
    "dedup",
    row_number().over(
        Window.partitionBy("id").orderBy(desc("modified_date"))
    )
)

df = df.filter(df.dedup == 1).drop("dedup")

df.createOrReplaceTempView("srctemp")

df = spark.sql(
    """
    SELECT *,
           current_timestamp() AS startTime,
           CAST('9999-12-31 23:59:59' AS timestamp) AS endTime,
           'Y' AS is_active
    FROM srctemp
    """
)

df.createOrReplaceTempView("src")

In [0]:
%sql
select * from src

marking the updated records as expired

In [0]:
%sql
merge into pyspark_data.source.dimCustomers as trg
using src as src
on trg.id = src.id
and trg.is_active = 'Y'
when matched and
(src.email <> trg.email or src.city <> trg.city or src.country <> trg.country or src.modified_date <> trg.modified_date)
then update set trg.end_time = current_timestamp(), trg.is_active = 'N'


merge-2 inserting new and updated records

In [0]:
%sql
MERGE INTO pyspark_data.source.DimCustomers AS trg
USING src AS src
ON trg.id = src.id
  AND trg.is_active = 'Y'
WHEN NOT MATCHED THEN INSERT (
  id,
  email,
  city,
  country,
  start_time,
  end_time,
  is_active
) VALUES (
  src.id,
  src.email,
  src.city,
  src.country,
  src.startTime,
  src.endTime,
  src.is_active
)

In [0]:
%sql
select * from pyspark_data.source.DimCustomers